# v1.5 Rashi-focus run (A100, ~7h, resumable)

Continues from the checkpoint-sweep winner with a rashi-weighted mixture:
synthetic Rashi-script data (random text — no language prior), upweighted real
rashi/tosafot crops, gemara/page replay against the seesaw. Trains at HIGH
resolution (min 3.2MP — small-script glyphs at ~2x), which inference-time
experiments showed only works if trained in.

Fill `WINNER_REVISION` from checkpoint_sweep.ipynb before running.
Secrets: `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
# Cell 1 — installs + env
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_PROJECT"] = "qwen-hebrew-finetune"
%pip install -q "unsloth[colab-new]" hf_transfer wandb

import torch
assert torch.cuda.is_available(), "No GPU — switch runtime to A100"

In [ ]:
# Cell 2 — data: talmud + synthetic rashi
from google.colab import userdata
from huggingface_hub import login, snapshot_download
from datasets import load_dataset

login(token=userdata.get("HF_TOKEN"))

talmud = load_dataset("isaacmg/talmud_finetune", split="train")
talmud_val = load_dataset("isaacmg/talmud_finetune", split="val")
synth = load_dataset("isaacmg/synthetic_rashi", split="train")
synth_eval = load_dataset("isaacmg/synthetic_rashi", split="eval")

crops = talmud.filter(lambda t: t == "crop_transcribe", input_columns="task")
pages = talmud.filter(lambda t: t == "page_extract", input_columns="task")
small_script = crops.filter(lambda s: s in ("rashi", "tosafot"), input_columns="section")
gemara = crops.filter(lambda s: s == "gemara", input_columns="section")
print(f"synth={len(synth)} small_script={len(small_script)} "
      f"gemara={len(gemara)} pages={len(pages)}")
assert len(synth) > 18000 and len(small_script) > 9000

In [ ]:
# Cell 3 — model at HIGH RESOLUTION + winner-checkpoint weights (atomic cell)
from unsloth import FastVisionModel
from transformers import AutoImageProcessor

MAX_SEQ = 12288
# Training-time upscale: small-script crops (~0.8MP) get ~2x linear
# magnification; inference-time upscaling alone was shown harmful, so this
# policy MUST also ship in the exported model's preprocessor_config.
MIN_PIX = 3_200_000
MAX_PIX = 4_500_000

CKPT_REPO_V1 = "isaacmg/qwen3-vl-8b-hebrew-ckpt"
WINNER_REVISION = "e2f85dde7d77c7117432737a7270296ccdc4a063"  # step 3800, sweep winner by mean CER

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ,
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", random_state=3407,
)

# weights-only warm start from the sweep winner (fresh optimizer + schedule)
assert len(WINNER_REVISION) == 40, "invalid revision SHA"
from safetensors.torch import load_file
from peft import set_peft_model_state_dict
local = snapshot_download(CKPT_REPO_V1, revision=WINNER_REVISION,
                          allow_patterns="last-checkpoint/adapter_model.safetensors")
missing = set_peft_model_state_dict(
    model, load_file(f"{local}/last-checkpoint/adapter_model.safetensors"))
print("unexpected keys:", len(getattr(missing, "unexpected_keys", [])))

tokenizer.image_processor = AutoImageProcessor.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct", min_pixels=MIN_PIX, max_pixels=MAX_PIX,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
lora_params = [n for n, p in model.named_parameters()
               if p.requires_grad and "lora" in n.lower()]
assert trainable > 0 and lora_params
print(f"trainable: {trainable/1e6:.1f}M ({len(lora_params)} LoRA tensors)")
print("resolution:", tokenizer.image_processor.size)


In [ ]:
# Cell 4 — conversation format + collator (native res preserved by resize='max')
from unsloth.trainer import UnslothVisionDataCollator

def to_conversation(sample):
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["question"]},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["answer"]},
            ]},
        ]
    }

collator = UnslothVisionDataCollator(
    model, tokenizer,
    formatting_func=to_conversation,
    resize="max",
    max_seq_length=MAX_SEQ,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# guardrail: one real batch must show upscaled pixels + masked labels
batch = collator([small_script[0], synth[0]])
pv = batch["pixel_values"]
assert pv is not None and pv.shape[0] > 20000, (
    f"{pv.shape[0]} patch rows — expected >20k for two ~3.2MP images; "
    "the resolution policy is not reaching the collator")
labels = batch["labels"]
unmasked = labels[0][labels[0] != -100]
_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
assert small_script[0]["answer"][:30] in _tok.decode(unmasked)
print(f"collator OK: pixel rows={pv.shape[0]}, labels masked")

In [ ]:
# Cell 5 — mixture + per-domain eval set + training (max_steps-bounded)
import wandb
from datasets import concatenate_datasets, interleave_datasets
from trl import SFTConfig, SFTTrainer
from unsloth import is_bf16_supported
import inspect

CKPT_REPO = "isaacmg/qwen3-vl-8b-hebrew-rashi-ckpt"
wandb.login(key=userdata.get("WANDB_API_KEY"))

mixture = interleave_datasets(
    [synth, small_script, gemara, pages],
    probabilities=[0.40, 0.30, 0.20, 0.10], seed=3407,
    stopping_strategy="all_exhausted",
)
# per-domain eval: talmud val crops + synthetic eval (val LOSS every 100 steps;
# the missing-eval lesson from the v1 run)
eval_ds = concatenate_datasets([
    talmud_val.filter(lambda t: t == "crop_transcribe", input_columns="task")
              .select(range(60)),
    synth_eval.select(range(60)),
])

def make_sft_config(**kw):
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in params:
        kw["max_length"] = kw.pop("max_seq_length")
    dropped = {k: kw.pop(k) for k in list(kw) if k not in params}
    if dropped:
        print(f"⚠️ dropped unsupported SFTConfig kwargs: {sorted(dropped)}")
    return SFTConfig(**kw)

def make_trainer(**kw):
    try:
        return SFTTrainer(**kw)
    except TypeError as e:
        if "tokenizer" in kw and ("tokenizer" in str(e) or "processing_class" in str(e)):
            kw["processing_class"] = kw.pop("tokenizer")
            return SFTTrainer(**kw)
        raise

FastVisionModel.for_training(model)
trainer = make_trainer(
    model=model, tokenizer=tokenizer, data_collator=collator,
    train_dataset=mixture, eval_dataset=eval_ds,
    args=make_sft_config(
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        max_steps=2000,                    # ~16k samples ≈ 7h; raise if stable
        learning_rate=5e-5,                # continuation LR
        warmup_ratio=0.02, lr_scheduler_type="cosine", weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
        save_steps=100, save_total_limit=2,
        push_to_hub=True, hub_model_id=CKPT_REPO,
        hub_strategy="checkpoint", hub_private_repo=True,
        optim="adamw_8bit", seed=3407, output_dir="outputs_rashi",
        report_to="wandb", run_name="rashi_focus_v15",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ,
    ),
)
trainer.train()

In [ ]:
# Cell 6 — export merged (policy-carrying) model
MERGED_REPO = "isaacmg/qwen3-vl-8b-hebrew-rashi-merged"
model.save_pretrained_merged("rashi-merged", tokenizer, save_method="merged_16bit")
# ship the TRAINING resolution policy with the model
tokenizer.image_processor.save_pretrained("rashi-merged")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", private=True)
print("pushed", MERGED_REPO)